In [3]:
from PIL import Image


WIDTH = 800
HEIGHT = 600


def edge(a, b, p):
    """
    判断点 p 在边 ab 的哪一侧。
    返回叉积结果。
    """
    return (p[0] - a[0]) * (b[1] - a[1]) - (p[1] - a[1]) * (b[0] - a[0])


def draw_triangle(img, v0, v1, v2, color):
    """
    在 framebuffer 上画一个实心三角形。
    v0/v1/v2 是屏幕坐标，例如 (100, 100)
    """

    # 1. 先求三角形包围盒，减少遍历范围
    min_x = max(int(min(v0[0], v1[0], v2[0])), 0)
    max_x = min(int(max(v0[0], v1[0], v2[0])), WIDTH - 1)
    min_y = max(int(min(v0[1], v1[1], v2[1])), 0)
    max_y = min(int(max(v0[1], v1[1], v2[1])), HEIGHT - 1)

    pixels = img.load()

    # 2. 遍历包围盒里的每个像素
    for y in range(min_y, max_y + 1):
        for x in range(min_x, max_x + 1):
            p = (x + 0.5, y + 0.5)

            # 3. 判断像素中心是否在三角形内部
            w0 = edge(v1, v2, p)
            w1 = edge(v2, v0, p)
            w2 = edge(v0, v1, p)

            # 同号说明在三角形内部
            if (w0 >= 0 and w1 >= 0 and w2 >= 0) or (w0 <= 0 and w1 <= 0 and w2 <= 0):
                pixels[x, y] = color


def main():
    # framebuffer
    img = Image.new("RGB", (WIDTH, HEIGHT), (20, 20, 30))

    # 顶点：这里先直接用屏幕坐标
    v0 = (400, 100)
    v1 = (150, 500)
    v2 = (650, 500)

    draw_triangle(img, v0, v1, v2, (255, 120, 60))

    img.save("output.png")
    print("saved output.png")


if __name__ == "__main__":
    main()

saved output.png


In [4]:
from PIL import Image

WIDTH = 800
HEIGHT = 600


def edge(a, b, p):
    return (p[0] - a[0]) * (b[1] - a[1]) - (p[1] - a[1]) * (b[0] - a[0])


def draw_triangle(img, v0, v1, v2, c0, c1, c2):
    min_x = max(int(min(v0[0], v1[0], v2[0])), 0)
    max_x = min(int(max(v0[0], v1[0], v2[0])), WIDTH - 1)
    min_y = max(int(min(v0[1], v1[1], v2[1])), 0)
    max_y = min(int(max(v0[1], v1[1], v2[1])), HEIGHT - 1)

    area = edge(v0, v1, v2)
    if area == 0:
        return

    pixels = img.load()

    for y in range(min_y, max_y + 1):
        for x in range(min_x, max_x + 1):
            p = (x + 0.5, y + 0.5)

            # 三个重心权重
            w0 = edge(v1, v2, p) / area
            w1 = edge(v2, v0, p) / area
            w2 = edge(v0, v1, p) / area

            if w0 >= 0 and w1 >= 0 and w2 >= 0:
                r = int(c0[0] * w0 + c1[0] * w1 + c2[0] * w2)
                g = int(c0[1] * w0 + c1[1] * w1 + c2[1] * w2)
                b = int(c0[2] * w0 + c1[2] * w1 + c2[2] * w2)

                pixels[x, y] = (r, g, b)


def main():
    img = Image.new("RGB", (WIDTH, HEIGHT), (20, 20, 30))

    v0 = (400, 100)
    v1 = (150, 500)
    v2 = (650, 500)

    c0 = (255, 60, 60)    # 顶点0：红
    c1 = (60, 255, 120)   # 顶点1：绿
    c2 = (80, 120, 255)   # 顶点2：蓝

    draw_triangle(img, v0, v1, v2, c0, c1, c2)

    img.save("output_gradient.png")
    print("saved output_gradient.png")


if __name__ == "__main__":
    main()

saved output_gradient.png


In [5]:
from PIL import Image
import math

WIDTH = 800
HEIGHT = 600


def edge(a, b, p):
    return (p[0] - a[0]) * (b[1] - a[1]) - (p[1] - a[1]) * (b[0] - a[0])


def draw_triangle(img, zbuffer, v0, v1, v2, c0, c1, c2):
    min_x = max(int(min(v0[0], v1[0], v2[0])), 0)
    max_x = min(int(max(v0[0], v1[0], v2[0])), WIDTH - 1)
    min_y = max(int(min(v0[1], v1[1], v2[1])), 0)
    max_y = min(int(max(v0[1], v1[1], v2[1])), HEIGHT - 1)

    area = edge(v0, v1, v2)
    if area == 0:
        return

    pixels = img.load()

    for y in range(min_y, max_y + 1):
        for x in range(min_x, max_x + 1):
            p = (x + 0.5, y + 0.5)

            w0 = edge(v1, v2, p) / area
            w1 = edge(v2, v0, p) / area
            w2 = edge(v0, v1, p) / area

            if w0 >= 0 and w1 >= 0 and w2 >= 0:
                # 插值深度。这里 z 越小表示越靠近摄像机
                z = v0[2] * w0 + v1[2] * w1 + v2[2] * w2

                if z < zbuffer[y][x]:
                    zbuffer[y][x] = z

                    r = int(c0[0] * w0 + c1[0] * w1 + c2[0] * w2)
                    g = int(c0[1] * w0 + c1[1] * w1 + c2[1] * w2)
                    b = int(c0[2] * w0 + c1[2] * w1 + c2[2] * w2)

                    pixels[x, y] = (r, g, b)


def main():
    img = Image.new("RGB", (WIDTH, HEIGHT), (20, 20, 30))

    # 初始深度为无穷远
    zbuffer = [[math.inf for _ in range(WIDTH)] for _ in range(HEIGHT)]

    # 三角形 A：远一点，z = 0.8
    a0 = (250, 150, 0.8)
    a1 = (100, 500, 0.8)
    a2 = (550, 500, 0.8)

    # 三角形 B：近一点，z = 0.3
    b0 = (500, 120, 0.3)
    b1 = (300, 520, 0.3)
    b2 = (700, 520, 0.3)

    draw_triangle(
        img, zbuffer,
        a0, a1, a2,
        (255, 80, 80), (255, 180, 80), (255, 80, 180)
    )

    draw_triangle(
        img, zbuffer,
        b0, b1, b2,
        (80, 180, 255), (80, 255, 180), (180, 80, 255)
    )

    img.save("output_zbuffer.png")
    print("saved output_zbuffer.png")


if __name__ == "__main__":
    main()

saved output_zbuffer.png


In [6]:
from PIL import Image
import math

WIDTH = 800
HEIGHT = 600


def edge(a, b, p):
    return (p[0] - a[0]) * (b[1] - a[1]) - (p[1] - a[1]) * (b[0] - a[0])


def sample_texture(texture, uv):
    u, v = uv

    # 限制到 0~1
    u = max(0.0, min(1.0, u))
    v = max(0.0, min(1.0, v))

    tx = int(u * (texture.width - 1))
    ty = int((1.0 - v) * (texture.height - 1))

    return texture.getpixel((tx, ty))[:3]


def draw_triangle(img, zbuffer, texture, v0, v1, v2, uv0, uv1, uv2):
    min_x = max(int(min(v0[0], v1[0], v2[0])), 0)
    max_x = min(int(max(v0[0], v1[0], v2[0])), WIDTH - 1)
    min_y = max(int(min(v0[1], v1[1], v2[1])), 0)
    max_y = min(int(max(v0[1], v1[1], v2[1])), HEIGHT - 1)

    area = edge(v0, v1, v2)
    if area == 0:
        return

    pixels = img.load()

    for y in range(min_y, max_y + 1):
        for x in range(min_x, max_x + 1):
            p = (x + 0.5, y + 0.5)

            w0 = edge(v1, v2, p) / area
            w1 = edge(v2, v0, p) / area
            w2 = edge(v0, v1, p) / area

            if w0 >= 0 and w1 >= 0 and w2 >= 0:
                z = v0[2] * w0 + v1[2] * w1 + v2[2] * w2

                if z < zbuffer[y][x]:
                    zbuffer[y][x] = z

                    # 插值 UV
                    u = uv0[0] * w0 + uv1[0] * w1 + uv2[0] * w2
                    v = uv0[1] * w0 + uv1[1] * w1 + uv2[1] * w2

                    color = sample_texture(texture, (u, v))
                    pixels[x, y] = color


def main():
    img = Image.new("RGB", (WIDTH, HEIGHT), (20, 20, 30))
    zbuffer = [[math.inf for _ in range(WIDTH)] for _ in range(HEIGHT)]

    texture = Image.open("texture.png").convert("RGB")

    v0 = (400, 100, 0.5)
    v1 = (150, 500, 0.5)
    v2 = (650, 500, 0.5)

    # 三个顶点对应纹理图片上的三个位置
    uv0 = (0.5, 1.0)
    uv1 = (0.0, 0.0)
    uv2 = (1.0, 0.0)

    draw_triangle(img, zbuffer, texture, v0, v1, v2, uv0, uv1, uv2)

    img.save("output_texture.png")
    print("saved output_texture.png")


if __name__ == "__main__":
    main()

saved output_texture.png


In [7]:
from PIL import Image
import math

WIDTH = 800
HEIGHT = 600


def normalize(v):
    length = math.sqrt(v[0] ** 2 + v[1] ** 2 + v[2] ** 2)
    if length == 0:
        return (0, 0, 0)
    return (v[0] / length, v[1] / length, v[2] / length)


def dot(a, b):
    return a[0] * b[0] + a[1] * b[1] + a[2] * b[2]


def edge(a, b, p):
    return (p[0] - a[0]) * (b[1] - a[1]) - (p[1] - a[1]) * (b[0] - a[0])


def sample_texture(texture, uv):
    u, v = uv
    u = max(0.0, min(1.0, u))
    v = max(0.0, min(1.0, v))

    tx = int(u * (texture.width - 1))
    ty = int((1.0 - v) * (texture.height - 1))

    return texture.getpixel((tx, ty))[:3]


def draw_triangle(img, zbuffer, texture, v0, v1, v2, uv0, uv1, uv2, n0, n1, n2, light_dir):
    min_x = max(int(min(v0[0], v1[0], v2[0])), 0)
    max_x = min(int(max(v0[0], v1[0], v2[0])), WIDTH - 1)
    min_y = max(int(min(v0[1], v1[1], v2[1])), 0)
    max_y = min(int(max(v0[1], v1[1], v2[1])), HEIGHT - 1)

    area = edge(v0, v1, v2)
    if area == 0:
        return

    pixels = img.load()

    for y in range(min_y, max_y + 1):
        for x in range(min_x, max_x + 1):
            p = (x + 0.5, y + 0.5)

            w0 = edge(v1, v2, p) / area
            w1 = edge(v2, v0, p) / area
            w2 = edge(v0, v1, p) / area

            if w0 >= 0 and w1 >= 0 and w2 >= 0:
                z = v0[2] * w0 + v1[2] * w1 + v2[2] * w2

                if z < zbuffer[y][x]:
                    zbuffer[y][x] = z

                    u = uv0[0] * w0 + uv1[0] * w1 + uv2[0] * w2
                    v = uv0[1] * w0 + uv1[1] * w1 + uv2[1] * w2

                    # 插值法线
                    normal = normalize((
                        n0[0] * w0 + n1[0] * w1 + n2[0] * w2,
                        n0[1] * w0 + n1[1] * w1 + n2[1] * w2,
                        n0[2] * w0 + n1[2] * w1 + n2[2] * w2,
                    ))

                    brightness = max(0.15, dot(normal, light_dir))

                    tex_color = sample_texture(texture, (u, v))

                    r = int(tex_color[0] * brightness)
                    g = int(tex_color[1] * brightness)
                    b = int(tex_color[2] * brightness)

                    pixels[x, y] = (r, g, b)


def main():
    img = Image.new("RGB", (WIDTH, HEIGHT), (20, 20, 30))
    zbuffer = [[math.inf for _ in range(WIDTH)] for _ in range(HEIGHT)]

    texture = Image.open("texture.png").convert("RGB")

    v0 = (400, 100, 0.5)
    v1 = (150, 500, 0.5)
    v2 = (650, 500, 0.5)

    uv0 = (0.5, 1.0)
    uv1 = (0.0, 0.0)
    uv2 = (1.0, 0.0)

    # 三个顶点的法线
    n0 = normalize((0.0, 0.0, 1.0))
    n1 = normalize((-0.6, 0.0, 1.0))
    n2 = normalize((0.6, 0.0, 1.0))

    light_dir = normalize((0.3, -0.5, 1.0))

    draw_triangle(
        img, zbuffer, texture,
        v0, v1, v2,
        uv0, uv1, uv2,
        n0, n1, n2,
        light_dir
    )

    img.save("output_light.png")
    print("saved output_light.png")


if __name__ == "__main__":
    main()

saved output_light.png


In [8]:
def mat4_mul_vec4(m, v):
    return (
        m[0][0]*v[0] + m[0][1]*v[1] + m[0][2]*v[2] + m[0][3]*v[3],
        m[1][0]*v[0] + m[1][1]*v[1] + m[1][2]*v[2] + m[1][3]*v[3],
        m[2][0]*v[0] + m[2][1]*v[1] + m[2][2]*v[2] + m[2][3]*v[3],
        m[3][0]*v[0] + m[3][1]*v[1] + m[3][2]*v[2] + m[3][3]*v[3],
    )


def perspective(fov_deg, aspect, near, far):
    f = 1.0 / math.tan(math.radians(fov_deg) / 2.0)
    return [
        [f / aspect, 0, 0, 0],
        [0, f, 0, 0],
        [0, 0, (far + near) / (near - far), (2 * far * near) / (near - far)],
        [0, 0, -1, 0],
    ]


def project_vertex(v, proj):
    x, y, z = v

    clip = mat4_mul_vec4(proj, (x, y, z, 1.0))

    # Perspective divide
    ndc_x = clip[0] / clip[3]
    ndc_y = clip[1] / clip[3]
    ndc_z = clip[2] / clip[3]

    # NDC [-1,1] → screen
    sx = (ndc_x * 0.5 + 0.5) * WIDTH
    sy = (1.0 - (ndc_y * 0.5 + 0.5)) * HEIGHT

    # z 转成 0~1，越小越近
    sz = ndc_z * 0.5 + 0.5

    return (sx, sy, sz)

In [9]:
from PIL import Image
import math

WIDTH = 800
HEIGHT = 600


def normalize(v):
    length = math.sqrt(v[0] ** 2 + v[1] ** 2 + v[2] ** 2)
    if length == 0:
        return (0, 0, 0)
    return (v[0] / length, v[1] / length, v[2] / length)


def dot(a, b):
    return a[0] * b[0] + a[1] * b[1] + a[2] * b[2]


def edge(a, b, p):
    return (p[0] - a[0]) * (b[1] - a[1]) - (p[1] - a[1]) * (b[0] - a[0])


def sample_texture(texture, uv):
    u, v = uv
    u = max(0.0, min(1.0, u))
    v = max(0.0, min(1.0, v))

    tx = int(u * (texture.width - 1))
    ty = int((1.0 - v) * (texture.height - 1))

    return texture.getpixel((tx, ty))[:3]


def draw_triangle(img, zbuffer, texture, v0, v1, v2, uv0, uv1, uv2, n0, n1, n2, light_dir):
    min_x = max(int(min(v0[0], v1[0], v2[0])), 0)
    max_x = min(int(max(v0[0], v1[0], v2[0])), WIDTH - 1)
    min_y = max(int(min(v0[1], v1[1], v2[1])), 0)
    max_y = min(int(max(v0[1], v1[1], v2[1])), HEIGHT - 1)

    area = edge(v0, v1, v2)
    if area == 0:
        return

    pixels = img.load()

    for y in range(min_y, max_y + 1):
        for x in range(min_x, max_x + 1):
            p = (x + 0.5, y + 0.5)

            w0 = edge(v1, v2, p) / area
            w1 = edge(v2, v0, p) / area
            w2 = edge(v0, v1, p) / area

            if w0 >= 0 and w1 >= 0 and w2 >= 0:
                z = v0[2] * w0 + v1[2] * w1 + v2[2] * w2

                if z < zbuffer[y][x]:
                    zbuffer[y][x] = z

                    u = uv0[0] * w0 + uv1[0] * w1 + uv2[0] * w2
                    v = uv0[1] * w0 + uv1[1] * w1 + uv2[1] * w2

                    # 插值法线
                    normal = normalize((
                        n0[0] * w0 + n1[0] * w1 + n2[0] * w2,
                        n0[1] * w0 + n1[1] * w1 + n2[1] * w2,
                        n0[2] * w0 + n1[2] * w1 + n2[2] * w2,
                    ))

                    brightness = max(0.15, dot(normal, light_dir))

                    tex_color = sample_texture(texture, (u, v))

                    r = int(tex_color[0] * brightness)
                    g = int(tex_color[1] * brightness)
                    b = int(tex_color[2] * brightness)

                    pixels[x, y] = (r, g, b)


def main():
    img = Image.new("RGB", (WIDTH, HEIGHT), (20, 20, 30))
    zbuffer = [[math.inf for _ in range(WIDTH)] for _ in range(HEIGHT)]

    texture = Image.open("texture.png").convert("RGB")

    proj = perspective(
        fov_deg=60,
        aspect=WIDTH / HEIGHT,
        near=0.1,
        far=100.0
    )

    # 3D 坐标，z 是负数，表示在摄像机前方
    p0 = (0.0, 1.0, -2.0)
    p1 = (-1.2, -1.0, -3.0)
    p2 = (1.2, -1.0, -3.0)

    v0 = project_vertex(p0, proj)
    v1 = project_vertex(p1, proj)
    v2 = project_vertex(p2, proj)

    uv0 = (0.5, 1.0)
    uv1 = (0.0, 0.0)
    uv2 = (1.0, 0.0)

    # 三个顶点的法线
    n0 = normalize((0.0, 0.0, 1.0))
    n1 = normalize((-0.6, 0.0, 1.0))
    n2 = normalize((0.6, 0.0, 1.0))

    light_dir = normalize((0.3, -0.5, 1.0))

    draw_triangle(
        img, zbuffer, texture,
        v0, v1, v2,
        uv0, uv1, uv2,
        n0, n1, n2,
        light_dir
    )

    img.save("output_light2.png")
    print("saved output_light2.png")


if __name__ == "__main__":
    main()

saved output_light2.png
